# Between-group differences in improvement — statistical validation

For every figure in the article the question is the same: **is the improvement (or outcome)
*different between the groups the figure shows*, how big is that difference, and which
contrast is the largest?**  Whether an improvement exists pre→post is assumed — we test only
whether groups *differ* in it.

Each figure → one between-group test on the relevant metric:

| Figure family | Metric compared between groups | Test |
|---|---|---|
| `pre_post_*` | **raw gain** (post − pre) | 2 groups: Mann–Whitney U · k groups: Kruskal–Wallis |
| `pre_post_trad_scale` | **grade-level change** (ordinal) | Kruskal–Wallis |
| `hake_gain_*` | **Hake normalized gain** | Mann–Whitney U / Kruskal–Wallis |
| `mathew_*` | raw gain, **High vs Low** starters | Mann–Whitney U (gap pre→post) |
| `mejora_*` | **improvement category mix** | χ² of independence |
| `self_confidence_comparison` | **self-efficacy gain** (ta post − pre) | Mann–Whitney U |
| `tcc_*` | **post cognitive load** (no pre exists) | Mann–Whitney U / Kruskal–Wallis |

**Every result reports:** the p-value, an **effect size** with magnitude, **which group
improved most**, and for k-group figures the **largest pairwise contrast** (Dunn-style
pairwise Mann–Whitney, Holm-adjusted). Effect sizes are stored on a common 0–1 `strength`
scale (|rank-biserial *r*| of the biggest contrast, or Cramér's V) so **§8 ranks all figures
against each other** — telling you where the biggest between-group differences are.

---
## Setup — identical data loading & filtering to `paper_results.ipynb`

In [1]:
import os, sys
import polars as pl

script_path = os.getcwd()
project_path = os.path.join(script_path, '..', '..', '..')
data_path = os.path.join(project_path, 'data', 'combined',
                         'processed_forms_interactions_data.parquet')
sys.path.append(project_path)
from src.utils.educational_impact.educational_impact_analysis import rename_df

forms_interactions_df = rename_df(pl.read_parquet(data_path))

problematic_ids = forms_interactions_df.filter(
    pl.col('grupo') == 'experimental', pl.col('chat_freq_use').is_null())['id'].to_list()
q25 = forms_interactions_df['score_tc_units_hake_gain'].quantile(0.25)
q75 = forms_interactions_df['score_tc_units_hake_gain'].quantile(0.75)
lb = q25 - 1.5 * (q75 - q25)
problematic_ids += forms_interactions_df.filter(pl.col('score_tc_units_hake_gain') < lb)['id'].to_list()
problematic_ids += forms_interactions_df.filter(pl.col('score_tc_pre') == 1)['id'].to_list()
problematic_ids = list(set(problematic_ids))

forms_interactions_df = forms_interactions_df.filter(~pl.col('id').is_in(problematic_ids))
no_efecto_techo_df = forms_interactions_df.filter(pl.col('score_tc_pre') < 0.8)

print("Analysis sample:", forms_interactions_df.shape)
print(forms_interactions_df['grupo'].value_counts())

Analysis sample: (183, 110)
shape: (2, 2)
┌──────────────┬───────┐
│ grupo        ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ experimental ┆ 88    │
│ control      ┆ 95    │
└──────────────┴───────┘


### Validation helpers

In [2]:
"""
Between-group comparison of the IMPROVEMENT / outcome for each figure.

The question is never "did scores rise pre→post" (that is assumed). It is always:
**does the size of the improvement differ BETWEEN the groups the figure shows, how big is
that difference, and which contrast is the largest?**  Every figure therefore reduces to a
between-group comparison of one metric:

  * pre_post_*      -> raw gain      (post - pre)
  * hake_gain_*     -> Hake gain     (already a column)
  * self_confidence -> self-eff gain (ta_post - ta_pre)
  * trad_scale      -> grade-level change (ordinal)
  * tcc_*           -> post load     (no pre exists for cognitive load)
  * mejora_*        -> improvement CATEGORY  (Improve / Not / Worsen)

Tests:
  * 2 groups      -> Mann-Whitney U            + rank-biserial r
  * k > 2 groups  -> Kruskal-Wallis (+ Dunn-style pairwise Mann-Whitney, Holm)
                     -> reports the LARGEST pairwise difference and the top group
  * categorical   -> Chi-square                + Cramér's V (+ best/worst group)

Every result stores a common 0–1 `strength` (|rank-biserial| of the biggest contrast, or
Cramér's V) so all figures can be ranked against each other at the end.
"""
from __future__ import annotations
import itertools
import numpy as np
import polars as pl
from scipy import stats
from statsmodels.stats.multitest import multipletests

ALPHA = 0.05
RESULTS: list[dict] = []


def reset_results():
    RESULTS.clear()


# ------------------------------------------------------------------ utilities
def _with_metric(df, metric=None, pre=None, post=None):
    if pre is not None and post is not None:
        return df.with_columns(
            (pl.col(post).cast(pl.Float64, strict=False) -
             pl.col(pre).cast(pl.Float64, strict=False)).alias("_m"))
    return df.with_columns(pl.col(metric).cast(pl.Float64, strict=False).alias("_m"))


def _num(df, col, filt=None):
    d = df if filt is None else df.filter(filt)
    a = d.select(pl.col(col).cast(pl.Float64, strict=False)).to_series().to_numpy()
    return a[~np.isnan(a)]


def _sig(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return " "
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."


def _mag_r(r):
    a = abs(r)
    return "negligible" if a < 0.1 else "small" if a < 0.3 else "medium" if a < 0.5 else "large"


def _mag_eps(e):
    return "negligible" if e < 0.01 else "small" if e < 0.06 else "medium" if e < 0.14 else "large"


def _rb(a, b):
    """rank-biserial for (a vs b): >0 means a stochastically larger than b."""
    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return 2 * U / (len(a) * len(b)) - 1, p


def _reg(**kw):
    RESULTS.append(kw)


# =====================================================================
# 2 groups  (e.g. experimental vs control)
# =====================================================================
def diff_2(figure, grouping, metric_label, df, group, g1, g2,
           metric=None, pre=None, post=None, baseline=False):
    d = _with_metric(df, metric, pre, post)
    a = _num(d, "_m", pl.col(group) == g1)
    b = _num(d, "_m", pl.col(group) == g2)
    if len(a) < 3 or len(b) < 3:
        print(f"█ {figure}: [skip] insufficient data (n={len(a)},{len(b)})\n")
        return
    rb, p = _rb(a, b)
    m1, m2 = float(np.median(a)), float(np.median(b))
    higher = g1 if rb > 0 else g2
    mag = _mag_r(rb)
    print(f"█ {figure}   ·  grouping: {group}   ·  metric: {metric_label}")
    if baseline:
        print(f"   Q: are {g1} and {g2} balanced at baseline? (want NO difference)")
    else:
        print(f"   Q: is the improvement different between {g1} and {g2}?")
    print(f"   Mann-Whitney U   n={len(a)+len(b)}   "
          f"median {g1}={m1:+.3f} vs {g2}={m2:+.3f}  (Δ={m1-m2:+.3f})")
    print(f"   p = {p:.4f} {_sig(p)}   rank-biserial r = {rb:+.3f} ({mag})")
    if baseline:
        verdict = ("⚠ groups DIFFER at baseline — potential confound." if p < ALPHA
                   else "baseline balanced (no significant difference).")
    else:
        verdict = (f"{higher} is higher on {metric_label} — difference is {mag}." if p < ALPHA
                   else f"no significant between-group difference ({mag} effect).")
    print(f"   → {verdict}\n")
    _reg(figure=figure, grouping=group, metric=metric_label, test="Mann-Whitney U",
         n=len(a) + len(b), p=p, effect_name="rank-biserial r", effect_value=rb,
         magnitude=mag, strength=abs(rb), biggest_contrast=f"{g1} vs {g2}",
         top_group=higher)


# =====================================================================
# k > 2 groups  (v4, v3, mejora-as-group ...)  + post-hoc "biggest difference"
# =====================================================================
def diff_k(figure, grouping, metric_label, df, group, order=None,
           metric=None, pre=None, post=None):
    d = _with_metric(df, metric, pre, post)
    if order is not None:
        present = set(d.select(pl.col(group)).drop_nulls().unique().to_series().to_list())
        cats = [c for c in order if c in present]
    else:
        cats = d.select(pl.col(group)).drop_nulls().unique().to_series().to_list()
    groups, labels = [], []
    for c in cats:
        arr = _num(d, "_m", pl.col(group) == c)
        if len(arr) >= 3:
            groups.append(arr)
            labels.append(c)
    if len(groups) < 2:
        print(f"█ {figure}: [skip] <2 usable groups\n")
        return
    H, p = stats.kruskal(*groups)
    n = sum(len(g) for g in groups)
    k = len(groups)
    eps = (H - k + 1) / (n - k) if (n - k) > 0 else np.nan
    meds = {l: float(np.median(g)) for l, g in zip(labels, groups)}
    top = max(meds, key=meds.get)
    bot = min(meds, key=meds.get)

    # pairwise Mann-Whitney (Holm) -> find the biggest contrast by |rank-biserial|
    pairs = list(itertools.combinations(range(k), 2))
    rows, praw = [], []
    for i, j in pairs:
        rb, pp = _rb(groups[i], groups[j])
        rows.append((labels[i], labels[j], rb, pp))
        praw.append(pp)
    padj = multipletests(praw, method="holm")[1] if praw else []
    biggest = max(range(len(rows)), key=lambda t: abs(rows[t][2])) if rows else None

    print(f"█ {figure}   ·  grouping: {group}   ·  metric: {metric_label}")
    print(f"   Q: does {metric_label} differ across the {k} groups?")
    print(f"   Kruskal-Wallis   H={H:.3f}, df={k-1}, n={n}   "
          f"p = {p:.4f} {_sig(p)}   ε² = {eps:.3f} ({_mag_eps(eps)})")
    print(f"   group medians: " + ", ".join(f"{l}={v:+.3f}" for l, v in meds.items()))
    print(f"   highest: {top} ({meds[top]:+.3f})   |   lowest: {bot} ({meds[bot]:+.3f})")
    if biggest is not None:
        c1, c2, rb, _ = rows[biggest]
        print(f"   BIGGEST pairwise difference: {c1} vs {c2}  "
              f"r={rb:+.3f} ({_mag_r(rb)}), p-Holm={padj[biggest]:.4f} {_sig(padj[biggest])}")
        sig_pairs = [(rows[t][0], rows[t][1], rows[t][2], padj[t])
                     for t in range(len(rows)) if padj[t] < ALPHA]
        if sig_pairs:
            print("   significant pairs (Holm): " +
                  "; ".join(f"{a}>{b} r={r:+.2f}" if r > 0 else f"{b}>{a} r={-r:+.2f}"
                            for a, b, r, _ in sig_pairs))
        else:
            print("   (no pair survives Holm correction)")
    print()
    c1, c2, rb, _ = rows[biggest]
    _reg(figure=figure, grouping=group, metric=metric_label, test="Kruskal-Wallis",
         n=n, p=p, effect_name="epsilon^2", effect_value=eps, magnitude=_mag_eps(eps),
         strength=abs(rb), biggest_contrast=f"{c1} vs {c2} (r={rb:+.2f})", top_group=top)


# =====================================================================
# categorical improvement (mejora)  -> proportions differ between groups?
# =====================================================================
def _cramers_v(chi2, table):
    n = table.sum()
    r, c = table.shape
    denom = n * (min(r, c) - 1)
    return np.sqrt(chi2 / denom) if denom > 0 else np.nan


def diff_cat(figure, grouping, df, cat, group, positive="Improve", order=None):
    sub = df.select([pl.col(group).cast(pl.Utf8).alias("_g"),
                     pl.col(cat).cast(pl.Utf8).alias("_c")]).drop_nulls()
    g_levels = sorted(sub["_g"].unique().to_list())
    c_levels = ([c for c in order if c in set(sub["_c"].to_list())]
                if order else sorted(sub["_c"].unique().to_list()))
    if sub.height == 0 or len(g_levels) < 2 or len(c_levels) < 2:
        print(f"█ {figure}: [skip] degenerate table\n")
        return
    counts = sub.group_by(["_g", "_c"]).len()
    look = {(r["_g"], r["_c"]): r["len"] for r in counts.iter_rows(named=True)}
    table = np.array([[look.get((g, c), 0) for c in c_levels] for g in g_levels], float)
    chi2, p, dof, exp = stats.chi2_contingency(table)
    v = _cramers_v(chi2, table)
    # % of `positive` category per group -> best / worst improving group
    pos_rate = {}
    if positive in c_levels:
        pi = c_levels.index(positive)
        for gi, g in enumerate(g_levels):
            tot = table[gi].sum()
            pos_rate[g] = table[gi, pi] / tot if tot else np.nan
    best = max(pos_rate, key=pos_rate.get) if pos_rate else "-"
    worst = min(pos_rate, key=pos_rate.get) if pos_rate else "-"
    small = (exp < 5).mean()

    print(f"█ {figure}   ·  grouping: {group}   ·  metric: {cat} (proportions)")
    print(f"   Q: does the improvement-category mix differ between groups?")
    print(f"   Chi-square   χ²={chi2:.3f}, df={dof}, n={int(table.sum())}   "
          f"p = {p:.4f} {_sig(p)}   Cramér's V = {v:.3f} ({_mag_r(v)})")
    if pos_rate:
        print("   %" + positive + " by group: " +
              ", ".join(f"{g}={pos_rate[g]*100:.0f}%" for g in g_levels))
        print(f"   most improving: {best} ({pos_rate[best]*100:.0f}%)   |   "
              f"least: {worst} ({pos_rate[worst]*100:.0f}%)")
    if small > 0:
        print(f"   note: {small*100:.0f}% of cells expected <5 (χ² approximate)")
    verdict = ("Proportions differ between groups." if p < ALPHA
               else "No significant difference in the improvement mix.")
    print(f"   → {verdict}\n")
    _reg(figure=figure, grouping=group, metric=f"{cat} proportions",
         test="Chi-square", n=int(table.sum()), p=p, effect_name="Cramér's V",
         effect_value=v, magnitude=_mag_r(v), strength=v,
         biggest_contrast=f"{best} vs {worst} (%{positive})", top_group=best)


# =====================================================================
# Matthew: within an arm, is improvement different for High vs Low starters?
# =====================================================================
def matthew(figure, df, pre, post, low_max=0.5, high_min=0.7):
    d = df.with_columns(
        pl.when(pl.col(pre) <= low_max).then(pl.lit("Low"))
          .when(pl.col(pre) >= high_min).then(pl.lit("High")).alias("_lvl"),
        (pl.col(post) - pl.col(pre)).alias("_m"))
    gh = _num(d, "_m", pl.col("_lvl") == "High")
    gl = _num(d, "_m", pl.col("_lvl") == "Low")
    if len(gh) < 3 or len(gl) < 3:
        print(f"█ {figure}: [skip] insufficient High/Low data\n")
        return
    rb, p = _rb(gh, gl)   # >0 => High gained more
    gap_pre = float(np.median(_num(d, pre, pl.col("_lvl") == "High")) -
                    np.median(_num(d, pre, pl.col("_lvl") == "Low")))
    gap_post = float(np.median(_num(d, post, pl.col("_lvl") == "High")) -
                     np.median(_num(d, post, pl.col("_lvl") == "Low")))
    mag = _mag_r(rb)
    if p < ALPHA:
        verdict = ("High starters gain MORE → gap widens (Matthew effect)." if rb > 0
                   else "Low starters gain MORE → gap narrows (compensatory).")
    else:
        verdict = "No significant difference in gain between High and Low starters."
    print(f"█ {figure}   ·  grouping: High vs Low baseline   ·  metric: raw gain")
    print(f"   Q: is the improvement different for High vs Low starters?")
    print(f"   Mann-Whitney U   n={len(gh)+len(gl)}   "
          f"gain High={np.median(gh):+.3f} vs Low={np.median(gl):+.3f}")
    print(f"   gap High−Low: pre={gap_pre:+.3f} → post={gap_post:+.3f}")
    print(f"   p = {p:.4f} {_sig(p)}   rank-biserial r = {rb:+.3f} ({mag})")
    print(f"   → {verdict}\n")
    _reg(figure=figure, grouping="High vs Low", metric="raw gain",
         test="Mann-Whitney U", n=len(gh) + len(gl), p=p,
         effect_name="rank-biserial r", effect_value=rb, magnitude=mag,
         strength=abs(rb), biggest_contrast="High vs Low",
         top_group="High" if rb > 0 else "Low")


# =====================================================================
# cross-figure summary — ranked by effect size ("biggest difference")
# =====================================================================
def summary():
    import pandas as pd
    if not RESULTS:
        print("No results.")
        return None
    df = pd.DataFrame(RESULTS)
    df["p_holm"] = multipletests(df["p"].values, method="holm")[1]
    df = df.sort_values("strength", ascending=False).reset_index(drop=True)
    show = df[["figure", "grouping", "test", "n", "p", "p_holm", "effect_name",
               "effect_value", "magnitude", "strength", "biggest_contrast", "top_group"]]
    with pd.option_context("display.max_rows", None, "display.width", 240,
                           "display.max_colwidth", 40,
                           "display.float_format", lambda v: f"{v:.4f}"):
        print("Ranked by between-group effect size (strength = |r| of biggest contrast, or V):\n")
        print(show.to_string(index=False))
    sig = df[df["p"] < ALPHA]
    print("\n" + "=" * 70)
    print("BIGGEST between-group differences (significant, by effect size):")
    for _, r in sig.head(8).iterrows():
        print(f"  • {r['figure']:<42} {r['magnitude']:>10}  "
              f"strength={r['strength']:.3f}  [{r['biggest_contrast']}]")
    if sig.empty:
        print("  (none reached significance)")
    return df


In [3]:
reset_results()

---
## 1 · Baseline balance (`tc_pre.pdf`, `ta_pre.pdf`)

Not an improvement — a check that the arms start equal. **Want NO difference.**

In [4]:
diff_2("tc_pre.pdf", "grupo", "baseline knowledge", forms_interactions_df,
       "grupo", "experimental", "control", metric="score_tc_pre", baseline=True)

diff_2("ta_pre.pdf", "grupo", "baseline self-efficacy", forms_interactions_df,
       "grupo", "experimental", "control", metric="score_ta_pre", baseline=True)

█ tc_pre.pdf   ·  grouping: grupo   ·  metric: baseline knowledge
   Q: are experimental and control balanced at baseline? (want NO difference)
   Mann-Whitney U   n=183   median experimental=+0.600 vs control=+0.600  (Δ=+0.000)
   p = 0.6704 n.s.   rank-biserial r = +0.036 (negligible)
   → baseline balanced (no significant difference).

█ ta_pre.pdf   ·  grouping: grupo   ·  metric: baseline self-efficacy
   Q: are experimental and control balanced at baseline? (want NO difference)
   Mann-Whitney U   n=183   median experimental=+0.500 vs control=+0.430  (Δ=+0.070)
   p = 0.0003 ***   rank-biserial r = +0.307 (medium)
   → ⚠ groups DIFFER at baseline — potential confound.



---
## 2 · Knowledge improvement (raw gain = post − pre) — does it differ between groups?

The `pre_post_*` figures: we compare the **raw gain** across the groups each one shows.

**`pre_post_comparison.pdf`** — experimental vs control.

In [5]:
diff_2("pre_post_comparison.pdf", "grupo", "raw gain (post−pre)", forms_interactions_df,
       "grupo", "experimental", "control", pre="score_tc_pre", post="score_tc_post")

█ pre_post_comparison.pdf   ·  grouping: grupo   ·  metric: raw gain (post−pre)
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.100 vs control=+0.200  (Δ=-0.100)
   p = 0.1690 n.s.   rank-biserial r = -0.117 (small)
   → no significant between-group difference (small effect).



**`pre_post_quality.pdf`** — across interaction quality (`grupo_segmented_v4`).

In [6]:
diff_k("pre_post_quality.pdf", "v4", "raw gain (post−pre)", forms_interactions_df,
       "grupo_segmented_v4", pre="score_tc_pre", post="score_tc_post")

█ pre_post_quality.pdf   ·  grouping: grupo_segmented_v4   ·  metric: raw gain (post−pre)
   Q: does raw gain (post−pre) differ across the 4 groups?
   Kruskal-Wallis   H=2.886, df=3, n=183   p = 0.4095 n.s.   ε² = -0.001 (negligible)
   group medians: ExpNotUsed=+0.100, ExpA=+0.100, ExpB=+0.200, control=+0.200
   highest: ExpB (+0.200)   |   lowest: ExpNotUsed (+0.100)
   BIGGEST pairwise difference: ExpNotUsed vs control  r=-0.199 (small), p-Holm=0.6738 n.s.
   (no pair survives Holm correction)



**`pre_post_freq_quality.pdf`** — across freq×quality (`grupo_segmented_v3`).

In [7]:
diff_k("pre_post_freq_quality.pdf", "v3", "raw gain (post−pre)", forms_interactions_df,
       "grupo_segmented_v3", pre="score_tc_pre", post="score_tc_post")

█ pre_post_freq_quality.pdf   ·  grouping: grupo_segmented_v3   ·  metric: raw gain (post−pre)
   Q: does raw gain (post−pre) differ across the 6 groups?
   Kruskal-Wallis   H=4.739, df=5, n=183   p = 0.4485 n.s.   ε² = -0.001 (negligible)
   group medians: ExpAA=+0.100, ExpNotUsed=+0.100, control=+0.200, ExpBB=+0.200, ExpAB=+0.200, ExpBA=+0.200
   highest: control (+0.200)   |   lowest: ExpAA (+0.100)
   BIGGEST pairwise difference: ExpAA vs ExpBA  r=-0.365 (medium), p-Holm=1.0000 n.s.
   (no pair survives Holm correction)



**`pre_post_trad_scale.pdf`** — the improvement here is the **grade-level change**
(Excellent=4 … Fail=0, so Δ>0 = better grade). We compare that change across quality groups.

In [8]:
TRAD = ['Excellent', 'Very Good', 'Good', 'Pass', 'Fail']       # best → worst
gmap = {c: (len(TRAD) - 1 - i) for i, c in enumerate(TRAD)}      # Excellent=4 … Fail=0

df_grade = forms_interactions_df.with_columns(
    pl.col("score_tc_cat_trad_scale_pre").replace_strict(gmap, default=None).alias("_gpre"),
    pl.col("score_tc_cat_trad_scale_post").replace_strict(gmap, default=None).alias("_gpost"),
).with_columns((pl.col("_gpost") - pl.col("_gpre")).alias("grade_change"))

diff_k("pre_post_trad_scale.pdf", "v4", "grade-level change", df_grade,
       "grupo_segmented_v4", metric="grade_change")

█ pre_post_trad_scale.pdf   ·  grouping: grupo_segmented_v4   ·  metric: grade-level change
   Q: does grade-level change differ across the 4 groups?
   Kruskal-Wallis   H=1.287, df=3, n=183   p = 0.7321 n.s.   ε² = -0.010 (negligible)
   group medians: ExpB=+1.000, ExpA=+1.000, ExpNotUsed=+1.000, control=+1.000
   highest: ExpB (+1.000)   |   lowest: ExpB (+1.000)
   BIGGEST pairwise difference: ExpA vs ExpNotUsed  r=+0.110 (small), p-Holm=1.0000 n.s.
   (no pair survives Holm correction)



---
## 3 · Matthew effect — is the High–Low gap bigger at post? (`mathew_control`, `mathew_experimental`)

Within each arm we compare the **gain of High vs Low starters**. The gap widens only if High
gain > Low gain; the output prints the median gap at pre and at post.

In [9]:
matthew("mathew_control.pdf",
        forms_interactions_df.filter(pl.col("grupo") == "control"),
        "score_tc_pre", "score_tc_post")

matthew("mathew_experimental.pdf",
        forms_interactions_df.filter(pl.col("grupo") == "experimental"),
        "score_tc_pre", "score_tc_post")

█ mathew_control.pdf   ·  grouping: High vs Low baseline   ·  metric: raw gain
   Q: is the improvement different for High vs Low starters?
   Mann-Whitney U   n=70   gain High=+0.100 vs Low=+0.400
   gap High−Low: pre=+0.400 → post=+0.100
   p = 0.0000 ***   rank-biserial r = -0.715 (large)
   → Low starters gain MORE → gap narrows (compensatory).

█ mathew_experimental.pdf   ·  grouping: High vs Low baseline   ·  metric: raw gain
   Q: is the improvement different for High vs Low starters?
   Mann-Whitney U   n=60   gain High=+0.050 vs Low=+0.300
   gap High−Low: pre=+0.350 → post=+0.100
   p = 0.0000 ***   rank-biserial r = -0.882 (large)
   → Low starters gain MORE → gap narrows (compensatory).



---
## 4 · Normalized (Hake) gain — does it differ between groups?

The Hake gain corrects for starting level; this is the cleanest improvement metric.

**`hake_gain_comparison.pdf`** — experimental vs control  *(headline)*.

In [10]:
diff_2("hake_gain_comparison.pdf", "grupo", "Hake gain", forms_interactions_df,
       "grupo", "experimental", "control", metric="score_tc_hake_gain")

█ hake_gain_comparison.pdf   ·  grouping: grupo   ·  metric: Hake gain
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.400 vs control=+0.500  (Δ=-0.100)
   p = 0.0374 *   rank-biserial r = -0.178 (small)
   → control is higher on Hake gain — difference is small.



**`hake_gain_quality.pdf`** — across interaction quality (`grupo_segmented_v4`).

In [11]:
diff_k("hake_gain_quality.pdf", "v4", "Hake gain", forms_interactions_df,
       "grupo_segmented_v4", metric="score_tc_hake_gain")

█ hake_gain_quality.pdf   ·  grouping: grupo_segmented_v4   ·  metric: Hake gain
   Q: does Hake gain differ across the 4 groups?
   Kruskal-Wallis   H=5.918, df=3, n=183   p = 0.1157 n.s.   ε² = 0.016 (small)
   group medians: control=+0.500, ExpNotUsed=+0.333, ExpB=+0.367, ExpA=+0.500
   highest: control (+0.500)   |   lowest: ExpNotUsed (+0.333)
   BIGGEST pairwise difference: control vs ExpNotUsed  r=+0.253 (small), p-Holm=0.2693 n.s.
   (no pair survives Holm correction)



**`hake_gain_frq_quality.pdf`** — across freq×quality (`grupo_segmented_v3`).

In [12]:
diff_k("hake_gain_frq_quality.pdf", "v3", "Hake gain", forms_interactions_df,
       "grupo_segmented_v3", metric="score_tc_hake_gain")

█ hake_gain_frq_quality.pdf   ·  grouping: grupo_segmented_v3   ·  metric: Hake gain
   Q: does Hake gain differ across the 6 groups?
   Kruskal-Wallis   H=7.200, df=5, n=183   p = 0.2062 n.s.   ε² = 0.012 (small)
   group medians: control=+0.500, ExpAB=+0.367, ExpBB=+0.343, ExpAA=+0.416, ExpBA=+0.500, ExpNotUsed=+0.333
   highest: control (+0.500)   |   lowest: ExpNotUsed (+0.333)
   BIGGEST pairwise difference: ExpBA vs ExpNotUsed  r=+0.365 (medium), p-Holm=0.9135 n.s.
   (no pair survives Holm correction)



**`hake_gain_frq_quality_no_ceiling_effect.pdf`** — same, `pre < 0.8` only.

In [13]:
diff_k("hake_gain_frq_quality_no_ceiling_effect.pdf", "v3", "Hake gain", no_efecto_techo_df,
       "grupo_segmented_v3", metric="score_tc_hake_gain")

█ hake_gain_frq_quality_no_ceiling_effect.pdf   ·  grouping: grupo_segmented_v3   ·  metric: Hake gain
   Q: does Hake gain differ across the 6 groups?
   Kruskal-Wallis   H=5.448, df=5, n=145   p = 0.3636 n.s.   ε² = 0.003 (negligible)
   group medians: ExpBA=+0.500, ExpAA=+0.500, ExpBB=+0.343, control=+0.600, ExpNotUsed=+0.367, ExpAB=+0.400
   highest: control (+0.600)   |   lowest: ExpBB (+0.343)
   BIGGEST pairwise difference: ExpBA vs ExpBB  r=+0.300 (medium), p-Holm=1.0000 n.s.
   (no pair survives Holm correction)



---
## 5 · Improvement-category mix — do proportions differ between groups?

`mejora_hake_gain_v2` = Improve / Not Improve / Worsen. χ² of independence + Cramér's V;
the output also prints the **% Improve per group** and the best/worst group.

**`mejora_comparison.pdf`** — by arm (`grupo`).

In [14]:
diff_cat("mejora_comparison.pdf", "grupo", forms_interactions_df,
         "mejora_hake_gain_v2", "grupo", order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_comparison.pdf   ·  grouping: grupo   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ²=1.849, df=2, n=183   p = 0.3968 n.s.   Cramér's V = 0.101 (small)
   %Improve by group: control=78%, experimental=72%
   most improving: control (78%)   |   least: experimental (72%)
   → No significant difference in the improvement mix.



**`mejora_quality.pdf`** — by interaction quality (`grupo_segmented_v4`).

In [15]:
diff_cat("mejora_quality.pdf", "v4", forms_interactions_df,
         "mejora_hake_gain_v2", "grupo_segmented_v4", order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_quality.pdf   ·  grouping: grupo_segmented_v4   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ²=4.664, df=6, n=183   p = 0.5876 n.s.   Cramér's V = 0.113 (small)
   %Improve by group: ExpA=80%, ExpB=69%, ExpNotUsed=67%, control=78%
   most improving: ExpA (80%)   |   least: ExpNotUsed (67%)
   note: 50% of cells expected <5 (χ² approximate)
   → No significant difference in the improvement mix.



**`mejora_frq_quality.pdf`** — by freq×quality (`grupo_segmented_v3`).

In [16]:
diff_cat("mejora_frq_quality.pdf", "v3", forms_interactions_df,
         "mejora_hake_gain_v2", "grupo_segmented_v3", order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_frq_quality.pdf   ·  grouping: grupo_segmented_v3   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ²=12.829, df=10, n=183   p = 0.2334 n.s.   Cramér's V = 0.187 (small)
   %Improve by group: ExpAA=67%, ExpAB=67%, ExpBA=92%, ExpBB=75%, ExpNotUsed=67%, control=78%
   most improving: ExpBA (92%)   |   least: ExpAA (67%)
   note: 56% of cells expected <5 (χ² approximate)
   → No significant difference in the improvement mix.



**`mejora_frq_quality_ceiling_effect.pdf`** — by freq×quality, `pre < 0.8` subset.

In [17]:
diff_cat("mejora_frq_quality_ceiling_effect.pdf", "v3", no_efecto_techo_df,
         "mejora_hake_gain_v2", "grupo_segmented_v3", order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_frq_quality_ceiling_effect.pdf   ·  grouping: grupo_segmented_v3   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ²=7.803, df=10, n=145   p = 0.6481 n.s.   Cramér's V = 0.164 (small)
   %Improve by group: ExpAA=86%, ExpAB=71%, ExpBA=91%, ExpBB=80%, ExpNotUsed=75%, control=83%
   most improving: ExpBA (91%)   |   least: ExpAB (71%)
   note: 56% of cells expected <5 (χ² approximate)
   → No significant difference in the improvement mix.



**`mejora_nota_pre.pdf`** — by interaction quality, within each baseline stratum.

In [18]:
for stratum in ["Low", "Medium", "High"]:
    diff_cat(f"mejora_nota_pre.pdf [pre={stratum}]", "v4",
             forms_interactions_df.filter(pl.col("score_tc_cat_pre") == stratum),
             "mejora_hake_gain_v2", "grupo_segmented_v4",
             order=['Improve', 'Not Improve', 'Worsen'])

█ mejora_nota_pre.pdf [pre=Low]   ·  grouping: grupo_segmented_v4   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ²=4.986, df=6, n=63   p = 0.5456 n.s.   Cramér's V = 0.199 (small)
   %Improve by group: ExpA=100%, ExpB=100%, ExpNotUsed=89%, control=89%
   most improving: ExpA (100%)   |   least: control (89%)
   note: 67% of cells expected <5 (χ² approximate)
   → No significant difference in the improvement mix.

█ mejora_nota_pre.pdf [pre=Medium]   ·  grouping: grupo_segmented_v4   ·  metric: mejora_hake_gain_v2 (proportions)
   Q: does the improvement-category mix differ between groups?
   Chi-square   χ²=5.592, df=6, n=82   p = 0.4704 n.s.   Cramér's V = 0.185 (small)
   %Improve by group: ExpA=83%, ExpB=56%, ExpNotUsed=64%, control=78%
   most improving: ExpA (83%)   |   least: ExpB (56%)
   note: 50% of cells expected <5 (χ² approximate)
   → No significant difference in the improvement mix.

█ mejora_

---
## 6 · Self-efficacy improvement (`self_confidence_comparison.pdf`)

Compare the **self-efficacy gain** (ta post − pre) between experimental and control.

In [19]:
diff_2("self_confidence_comparison.pdf", "grupo", "self-efficacy gain (post−pre)",
       forms_interactions_df, "grupo", "experimental", "control",
       pre="score_ta_pre", post="score_ta_post")

█ self_confidence_comparison.pdf   ·  grouping: grupo   ·  metric: self-efficacy gain (post−pre)
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.050 vs control=+0.080  (Δ=-0.030)
   p = 0.0399 *   rank-biserial r = -0.176 (small)
   → control is higher on self-efficacy gain (post−pre) — difference is small.



---
## 7 · Cognitive load — does post load differ between groups?

Load is measured only post, so we compare the **post score** between groups (not an
improvement). Three subscales: relevant/germane, extraneous, intrinsic.

**`tcc_comparison.pdf`** — each subscale, experimental vs control.

In [20]:
for c, lab in [("score_tcc_rel_post", "relevant load"),
                  ("score_tcc_ext_post", "extraneous load"),
                  ("score_tcc_int_post", "intrinsic load")]:
    diff_2(f"tcc_comparison.pdf [{lab}]", "grupo", lab, forms_interactions_df,
           "grupo", "experimental", "control", metric=c)

█ tcc_comparison.pdf [relevant load]   ·  grouping: grupo   ·  metric: relevant load
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.730 vs control=+0.730  (Δ=+0.000)
   p = 0.3863 n.s.   rank-biserial r = +0.074 (negligible)
   → no significant between-group difference (negligible effect).

█ tcc_comparison.pdf [extraneous load]   ·  grouping: grupo   ·  metric: extraneous load
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.430 vs control=+0.300  (Δ=+0.130)
   p = 0.0002 ***   rank-biserial r = +0.324 (medium)
   → experimental is higher on extraneous load — difference is medium.

█ tcc_comparison.pdf [intrinsic load]   ·  grouping: grupo   ·  metric: intrinsic load
   Q: is the improvement different between experimental and control?
   Mann-Whitney U   n=183   median experimental=+0.415 vs control=+0.300  (Δ=+0.115)
   p = 0.0630 n.s.   rank

**`tcc_comparison_quality.pdf`** — each subscale across interaction quality (`grupo_segmented_v4`).

In [21]:
for c, lab in [("score_tcc_rel_post", "relevant load"),
                  ("score_tcc_ext_post", "extraneous load"),
                  ("score_tcc_int_post", "intrinsic load")]:
    diff_k(f"tcc_comparison_quality.pdf [{lab}]", "v4", lab, forms_interactions_df,
           "grupo_segmented_v4", metric=c)

█ tcc_comparison_quality.pdf [relevant load]   ·  grouping: grupo_segmented_v4   ·  metric: relevant load
   Q: does relevant load differ across the 4 groups?
   Kruskal-Wallis   H=5.731, df=3, n=183   p = 0.1255 n.s.   ε² = 0.015 (small)
   group medians: ExpB=+0.700, ExpNotUsed=+0.730, control=+0.730, ExpA=+0.830
   highest: ExpA (+0.830)   |   lowest: ExpB (+0.700)
   BIGGEST pairwise difference: ExpB vs ExpA  r=-0.302 (medium), p-Holm=0.2307 n.s.
   (no pair survives Holm correction)

█ tcc_comparison_quality.pdf [extraneous load]   ·  grouping: grupo_segmented_v4   ·  metric: extraneous load
   Q: does extraneous load differ across the 4 groups?
   Kruskal-Wallis   H=14.833, df=3, n=183   p = 0.0020 **   ε² = 0.066 (medium)
   group medians: ExpB=+0.430, ExpNotUsed=+0.430, ExpA=+0.430, control=+0.300
   highest: ExpB (+0.430)   |   lowest: control (+0.300)
   BIGGEST pairwise difference: ExpNotUsed vs control  r=+0.360 (medium), p-Holm=0.0216 *
   significant pairs (Holm): ExpB>co

**`tcc_mejora.pdf`** — each subscale across improvement outcome (`mejora_hake_gain_v2`).

In [22]:
for c, lab in [("score_tcc_rel_post", "relevant load"),
                  ("score_tcc_ext_post", "extraneous load"),
                  ("score_tcc_int_post", "intrinsic load")]:
    diff_k(f"tcc_mejora.pdf [{lab}]", "mejora", lab, forms_interactions_df,
           "mejora_hake_gain_v2", metric=c)

█ tcc_mejora.pdf [relevant load]   ·  grouping: mejora_hake_gain_v2   ·  metric: relevant load
   Q: does relevant load differ across the 3 groups?
   Kruskal-Wallis   H=11.160, df=2, n=183   p = 0.0038 **   ε² = 0.051 (small)
   group medians: Not Improve=+0.670, Improve=+0.770, Worsen=+0.670
   highest: Improve (+0.770)   |   lowest: Not Improve (+0.670)
   BIGGEST pairwise difference: Improve vs Worsen  r=+0.354 (medium), p-Holm=0.0147 *
   significant pairs (Holm): Improve>Worsen r=+0.35

█ tcc_mejora.pdf [extraneous load]   ·  grouping: mejora_hake_gain_v2   ·  metric: extraneous load
   Q: does extraneous load differ across the 3 groups?
   Kruskal-Wallis   H=19.389, df=2, n=183   p = 0.0001 ***   ε² = 0.097 (medium)
   group medians: Improve=+0.300, Not Improve=+0.430, Worsen=+0.500
   highest: Worsen (+0.500)   |   lowest: Improve (+0.300)
   BIGGEST pairwise difference: Improve vs Worsen  r=-0.526 (large), p-Holm=0.0001 ***
   significant pairs (Holm): Worsen>Improve r=+0.53



---
## 8 · Cross-figure ranking — where are the biggest between-group differences?

All figures ranked by effect size (`strength`: |rank-biserial *r*| of the biggest contrast,
or Cramér's V). This is the answer to *"which difference is biggest, across all the graphs."*
`p_holm` adds a family-wise Holm correction across every test.

In [23]:
ranking = summary()

Ranked by between-group effect size (strength = |r| of biggest contrast, or V):

                                      figure            grouping           test   n      p  p_holm     effect_name  effect_value  magnitude  strength                   biggest_contrast    top_group
                     mathew_experimental.pdf         High vs Low Mann-Whitney U  60 0.0000  0.0000 rank-biserial r       -0.8817      large    0.8817                        High vs Low          Low
                          mathew_control.pdf         High vs Low Mann-Whitney U  70 0.0000  0.0000 rank-biserial r       -0.7151      large    0.7151                        High vs Low          Low
            tcc_mejora.pdf [extraneous load] mejora_hake_gain_v2 Kruskal-Wallis 183 0.0001  0.0017       epsilon^2        0.0966     medium    0.5264        Improve vs Worsen (r=-0.53)       Worsen
                   pre_post_freq_quality.pdf  grupo_segmented_v3 Kruskal-Wallis 183 0.4485  1.0000       epsilon^2       -0.001

In [24]:
out = os.path.join(project_path, 'plots', 'between_group_differences.csv')
try:
    ranking.to_csv(out, index=False); print("Saved:", out)
except Exception as e:
    print("Could not save:", e)

Could not save: Cannot save file into a non-existent directory: '/home/miguel_ramos/MisArchivos/va-educational-impact/src/notebooks/paper/../../../plots'
